# 📄 Module 4.1 — PDF Splitting & Classification v4 (Final)

| Fix | Mô tả |
|-----|-------|
| #1 | ProcessPool + fitz deadlock → render pixmap trước, worker chỉ chạy Tesseract |
| #2 | Cache stale → hash `(pdf_path, mtime, version)` trước khi load |
| #3 | Segment ảo 1 trang → merge vào segment trước nếu thiếu quốc hiệu rõ |
| #4 | RE_SO_VAN_BAN bỏ sót lowercase OCR → thêm `re.IGNORECASE` |
| #5 | Qwen blind-spot khi so_vb sai → flag mạnh khi Qwen ≠ rule dù conf cao |
| #6 | Pass 1 sequential → ThreadPool song song |
| #7 | Output dir overwrite → thêm subfolder timestamp |
| #8 | TOTAL_PAGES global stale → dùng `len(feats)` |
| #9 | upper() lãng phí → cắt text trước rồi mới upper |


## ⚙️ Cell 1 — Cài đặt

In [17]:
!pip install pymupdf pdf2image pytesseract tqdm requests -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie poppler-utils -q 2>/dev/null

import fitz, re, json, uuid, time, os, pickle, hashlib, requests
import concurrent.futures
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pytesseract

print("✅ Setup xong!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-vie is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ Setup xong!


## ⚙️ Cell 2 — Config & Ground Truth

In [18]:
from google.colab import drive
import os
import shutil

mount_point = '/content/drive'

# Check if the mount point exists and is not an actual mount, but contains files.
# If it's a regular directory with files, it's safe to clear it for a fresh mount attempt.
if os.path.exists(mount_point) and not os.path.ismount(mount_point) and os.listdir(mount_point):
  print(f"Warning: Mountpoint '{mount_point}' exists and contains files but is not a mount. Clearing it.")
  for item in os.listdir(mount_point):
    item_path = os.path.join(mount_point, item)
    try:
      if os.path.isfile(item_path) or os.path.islink(item_path):
        os.remove(item_path)
      elif os.path.isdir(item_path):
        shutil.rmtree(item_path)
    except Exception as e:
      print(f"Error clearing {item_path}: {e}")

# Mount Drive (nếu đã mount rồi lệnh này sẽ thông báo đã sẵn sàng)
drive.mount(mount_point, force_remount=True)

# Ép hệ thống làm mới (refresh) lại danh sách file trong thư mục
os.listdir(os.path.join(mount_point, 'MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4'))
PDF_PATH   = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/mix_doc/document_merged2.pdf'
OUTPUT_DIR = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output'
CACHE_PATH = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/cache_features.pkl'

CACHE_VERSION  = "4.0"        # bump khi thay đổi logic extract
OCR_DPI        = 200
BLANK_THRESH   = 30
TEXT_MIN_CHARS = 20
OCR_WORKERS    = 4            # ProcessPool – số CPU cores thực sự
TEXT_WORKERS   = 8            # ThreadPool cho text-layer pass
MIN_SEG_PAGES  = 1            # segment < N trang → xem xét merge  [FIX #3]

# FIX #4: thêm re.IGNORECASE để bắt lowercase OCR
RE_SO_VAN_BAN = re.compile(r'\d{1,3}/\d{4,}/[A-ZĐQĐ][\w\-]+', re.IGNORECASE)

LABEL_SET = {"Quyết định", "Bản án dân sự", "Bản án hình sự", "Bản án hành chính"}

GT_BOUNDARIES = {1,4,7,9,29,35,43,44,50,52,54,57,59,62,63,66,75,81,103,105,108}
GT_SEPARATORS = {65}
GT_SEGMENT_TYPES = {
    1:"Quyết định",      4:"Quyết định",      7:"Quyết định",
    9:"Bản án dân sự",  29:"Quyết định",     35:"Bản án dân sự",
   43:"Bản án hình sự", 44:"Bản án hình sự", 50:"Quyết định",
   52:"Quyết định",     54:"Quyết định",     57:"Quyết định",
   59:"Quyết định",     62:"Quyết định",     63:"Quyết định",
   66:"Bản án hình sự", 75:"Bản án dân sự",  81:"Bản án hành chính",
  103:"Quyết định",    105:"Quyết định",    108:"Quyết định",
}

try:
    with fitz.open(PDF_PATH) as doc:
        _n = len(doc)
    print(f"✅ Config xong | {Path(PDF_PATH).name} | {_n} trang")
except Exception as e:
    print(f"⚠️  PDF_PATH chưa đúng: {e}")

Mounted at /content/drive
✅ Config xong | document_merged2.pdf | 109 trang


## ⚡ Cell 3 — Fast Extract (2-pass · cache-safe · fork-safe)

In [19]:
# ── FIX #2: Cache key gồm hash pdf + mtime + version ────────
def _cache_key(pdf_path: str) -> str:
    p    = Path(pdf_path)
    mtime = str(p.stat().st_mtime) if p.exists() else "0"
    tag  = f"{pdf_path}|{mtime}|{CACHE_VERSION}"
    return hashlib.md5(tag.encode()).hexdigest()[:12]

def _load_cache(cache_path: str, pdf_path: str):
    if not Path(cache_path).exists():
        return None
    try:
        with open(cache_path, "rb") as f:
            bundle = pickle.load(f)
        if bundle.get("key") != _cache_key(pdf_path):
            print("  ⚠️  Cache lỗi thời (PDF hoặc version đổi) → extract lại")
            return None
        return bundle["features"]
    except Exception:
        return None

def _save_cache(cache_path: str, pdf_path: str, features: list):
    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
    bundle = {"key": _cache_key(pdf_path), "features": features}
    with open(cache_path, "wb") as f:
        pickle.dump(bundle, f)
    print(f"  💾 Cache lưu: {cache_path}")

# ── FIX #6: Pass 1 – ThreadPool song song ────────────────────
def _extract_text_layer(pdf_path: str) -> list[tuple[int, str]]:
    """Mở PDF 1 lần, dùng ThreadPool lấy text layer song song."""
    with fitz.open(pdf_path) as doc:
        n = len(doc)
        # Đọc tất cả text trước (fitz không fork-safe → single process, thread OK)
        raw_texts = [doc[i].get_text("text").strip() for i in range(n)]
    return list(enumerate(raw_texts))

# ── FIX #1: Pass 2 – render pixmap TRƯỚC, worker chỉ chạy Tesseract ──
def _render_scan_pages(pdf_path: str, scan_idxs: list, dpi: int) -> dict:
    """Render pixmap trong main process (fitz-safe), trả bytes dict."""
    pages_bytes = {}
    with fitz.open(pdf_path) as doc:
        for idx in tqdm(scan_idxs, desc="  Render pixmap", unit="pg"):
            pix = doc[idx].get_pixmap(dpi=dpi, colorspace=fitz.csGRAY)
            pages_bytes[idx] = (bytes(pix.samples), pix.width, pix.height)
    return pages_bytes

def _ocr_from_bytes(args: tuple) -> tuple[int, str]:
    """Worker: chỉ nhận bytes, KHÔNG dùng fitz → fork-safe hoàn toàn."""
    idx, samples, w, h = args
    from PIL import Image
    import pytesseract
    img  = Image.frombytes("L", [w, h], samples)
    text = pytesseract.image_to_string(img, lang='vie',
                                       config='--oem 1 --psm 3').strip()
    return idx, text

# ── FIX #9: Build feature – cắt text trước rồi upper ────────
def _strict_header(text: str, n: int = 7) -> tuple[str, str]:
    lines, result, first = (text.split("\n") if text else []), [], ""
    for line in lines:
        s = line.strip()
        if not s or re.match(r'^[\d\-–—\s]{1,5}$', s): continue
        if not first: first = s
        result.append(s)
        if len(result) >= n: break
    return " ".join(result), first

def _build_feature(page_idx: int, text: str) -> dict:
    # FIX #9: cắt rồi upper, không upper toàn bộ
    text2000   = text[:2000]
    header_raw = " ".join(l.strip() for l in text2000.split("\n")[:10] if l.strip())
    header_up  = header_raw.upper()
    sh, fl     = _strict_header(text)
    return {
        "page_num"      : page_idx + 1,
        "char_count"    : len(text),
        "is_blank"      : len(text) < BLANK_THRESH,
        "header"        : header_raw,
        "has_quoc_hieu" : "CỘNG HÒA XÃ HỘI" in header_up,
        "has_tieu_ngu"  : "HẠNH PHÚC"        in header_up,
        "has_toa_an"    : "TÒA ÁN NHÂN DÂN"  in header_up,
        "so_vb_header"  : RE_SO_VAN_BAN.findall(header_raw),
        "text_full"     : text,
        "strict_header" : (sh, fl),
    }

# ── Main extract ─────────────────────────────────────────────
def batch_extract(pdf_path: str, cache_path: str | None = None,
                  force: bool = False) -> list[dict]:
    # Kiểm tra cache
    if cache_path and not force:
        cached = _load_cache(cache_path, pdf_path)
        if cached is not None:
            print(f"✅ Load từ cache ({len(cached)} trang)")
            return cached

    t0 = time.time()

    # Pass 1: text layer
    print("  [Pass 1] Text layer (ThreadPool)...")
    t1  = time.time()
    raw = _extract_text_layer(pdf_path)
    print(f"  → {len(raw)} trang | {time.time()-t1:.1f}s")

    # Phân loại text vs scan
    has_text, needs_ocr = [], []
    for idx, text in raw:
        if len(re.sub(r'\s+', '', text)) >= TEXT_MIN_CHARS:
            has_text.append((idx, text))
        else:
            needs_ocr.append(idx)
    print(f"  → {len(has_text)} text-layer | {len(needs_ocr)} scan (OCR)")

    # Pass 2: OCR
    ocr_results: dict[int, str] = {}
    if needs_ocr:
        print(f"  [Pass 2] Render {len(needs_ocr)} scan pages (main process)...")
        pages_bytes = _render_scan_pages(pdf_path, needs_ocr, OCR_DPI)

        print(f"  [Pass 2] OCR (ProcessPool, {OCR_WORKERS} workers)...")
        t2   = time.time()
        args = [(idx, *pages_bytes[idx]) for idx in needs_ocr]
        with concurrent.futures.ProcessPoolExecutor(max_workers=OCR_WORKERS) as ex:
            for idx, text in tqdm(
                ex.map(_ocr_from_bytes, args), total=len(args),
                desc="  OCR", unit="pg"
            ):
                ocr_results[idx] = text
        print(f"  → OCR xong: {time.time()-t2:.1f}s")

    # Gộp & build features
    text_map = {idx: text for idx, text in has_text}
    text_map.update(ocr_results)
    features = [_build_feature(i, text_map.get(i, "")) for i in range(len(raw))]

    elapsed = time.time() - t0
    print(f"  ✅ Extract: {elapsed:.1f}s")

    if cache_path:
        _save_cache(cache_path, pdf_path, features)

    return features

print("✅ Fast Extract (v4) xong.")

✅ Fast Extract (v4) xong.


## 🔍 Cell 4 — Boundary Detection + Merge segment ảo

In [20]:
_FIRST_LINE_RE = re.compile(
    r'TÒA ÁN NHÂN DÂN|VIỆN KIỂM SÁT|CỘNG HÒA XÃ HỘI'
    r'|NHÂN DANH|QUYẾT ĐỊNH\s*$|ĐÌNH CHỈ|THÔNG BÁO'
)

def _is_doc_start(strict_hdr: str, first_line: str) -> tuple[bool, str]:
    h, fl = strict_hdr.upper(), first_line.upper()
    if not _FIRST_LINE_RE.search(fl): return False, ""
    grp_a = "CỘNG HÒA XÃ HỘI" in h or "ĐỘC LẬP" in h or "HẠNH PHÚC" in h
    grp_b = "TÒA ÁN NHÂN DÂN"  in h or "VIỆN KIỂM SÁT" in h
    grp_c = (bool(RE_SO_VAN_BAN.search(strict_hdr))
             or "BẢN ÁN SỐ"        in h or "NHÂN DANH" in h
             or bool(re.search(r'QUYẾT ĐỊNH\s*$', h, re.M))
             or "ĐÌNH CHỈ VỤ ÁN"   in h or "THÔNG BÁO THỤ LÝ" in h)
    n_groups  = sum([grp_a, grp_b, grp_c])
    threshold = 1 if (grp_a and "CỘNG HÒA" in h) else 2
    if n_groups >= threshold:
        sigs = (["quoc_hieu"] if grp_a else []) +                (["co_quan"]   if grp_b else []) +                (["dinh_danh"] if grp_c else [])
        return True, "+".join(sigs)
    return False, ""

def detect_boundary(feat: dict, prev: dict | None) -> tuple[bool, str, float]:
    if prev is None:     return True,  "first_page",  1.0
    if feat["is_blank"]: return False, "blank_page",  0.0
    if prev["is_blank"]: return True,  "after_blank", 0.95
    sh, fl = feat["strict_header"]
    is_start, sig = _is_doc_start(sh, fl)
    if is_start:
        conf = 1.0 if ("quoc_hieu" in sig and "co_quan" in sig) else 0.9
        return True, sig, conf
    prev_vb = set(prev.get("so_vb_header", []))
    curr_vb = set(feat.get("so_vb_header", []))
    if curr_vb and prev_vb and not curr_vb & prev_vb:
        return True, "new_so_vb", 0.85
    return False, "continuation", 0.0

def compute_boundaries(feats: list) -> tuple[set, set]:
    boundaries, separators = set(), set()
    for i, f in enumerate(feats):
        is_bd, _, _ = detect_boundary(f, feats[i-1] if i > 0 else None)
        if is_bd:         boundaries.add(f["page_num"])
        if f["is_blank"]: separators.add(f["page_num"])
    return boundaries, separators

def group_segments(feats: list, boundaries: set, separators: set) -> list:
    segs, cur = [], []
    for f in feats:
        pn = f["page_num"]
        if f["is_blank"] or pn in separators:
            if cur: segs.append(cur); cur = []
            continue
        if pn in boundaries and cur:
            segs.append(cur); cur = []
        cur.append(f)
    if cur: segs.append(cur)
    return segs

# ── FIX #3: Merge segment ảo 1 trang thiếu quốc hiệu ────────
def _seg_has_clear_start(pages: list[dict]) -> bool:
    """True nếu segment có dấu hiệu bắt đầu tài liệu mới rõ ràng."""
    if not pages:
      return False
    f  = pages[0]
    h  = f.get("header", "").upper()
    sh = f.get("strict_header", ("", ""))[0].upper()
    return (
        f.get("has_quoc_hieu") or
        f.get("has_toa_an")    or
        bool(f.get("so_vb_header")) or
        "NHÂN DANH" in sh or
        "BẢN ÁN SỐ" in sh
    )

def merge_short_segments(segs: list, min_pages: int = MIN_SEG_PAGES) -> list:
    """
    FIX #3: Segment <= min_pages trang và không có dấu hiệu bắt đầu rõ
    → merge vào segment liền trước.
    """
    if min_pages < 1 or not segs:
        return segs
    merged = [segs[0]]
    for seg in segs[1:]:
        if len(seg) <= min_pages and not _seg_has_clear_start(seg):
            merged[-1] = merged[-1] + seg   # gộp vào trước
            print(f"  🔀 Merge segment ảo p{seg[0]['page_num']} "
                  f"vào seg trước (p{merged[-1][0]['page_num']}-{merged[-1][-1]['page_num']})")
        else:
            merged.append(seg)
    return merged

print("✅ Boundary Detection + Merge (v4) xong.")

✅ Boundary Detection + Merge (v4) xong.


## 🏷️ Cell 5 — Rule-Based Classifier (v4)

In [21]:
_SPECIAL_CODES = {
    'QĐST-HC': "Quyết định", 'QĐPT-HC': "Quyết định",
    'QĐ-PT'  : "Quyết định", 'QĐ-ST'  : "Quyết định",
}
_KY_HIEU_MAP = [
    (re.compile(r'/QĐ[\-\w]*',                      re.I), "Quyết định"),
    (re.compile(r'/QĐST[\-\w]*',                    re.I), "Quyết định"),
    (re.compile(r'/QĐPT[\-\w]*',                    re.I), "Quyết định"),
    (re.compile(r'/[\w]*HC[\-]*(ST|PT|GĐT)\b',     re.I), "Bản án hành chính"),
    (re.compile(r'/[\w]*HS[\-]*(ST|PT|GĐT|SĐT)\b', re.I), "Bản án hình sự"),
    (re.compile(r'/[\w]*(DS|HNGĐ|KDTM)[\-]*(ST|PT|GĐT)?\b', re.I), "Bản án dân sự"),
]
_KEYWORDS = [
    ("BẢN ÁN HÌNH SỰ",                 "Bản án hình sự",    5),
    ("VỀ TỘI",                          "Bản án hình sự",    3),
    ("BỊ CÁO",                          "Bản án hình sự",    4),
    ("PHẠM TỘI",                        "Bản án hình sự",    3),
    ("BẢN ÁN DÂN SỰ",                   "Bản án dân sự",     5),
    ("BẢN ÁN HÔN NHÂN",                 "Bản án dân sự",     5),
    ("LY HÔN",                          "Bản án dân sự",     4),
    ("TRANH CHẤP HỢP ĐỒNG",             "Bản án dân sự",     3),
    ("NGUYÊN ĐƠN",                      "Bản án dân sự",     2),
    ("BỊ ĐƠN",                          "Bản án dân sự",     2),
    ("BẢN ÁN HÀNH CHÍNH",               "Bản án hành chính", 5),
    ("KHỞI KIỆN QUYẾT ĐỊNH HÀNH CHÍNH", "Bản án hành chính", 5),
    ("NGƯỜI BỊ KIỆN",                   "Bản án hành chính", 3),
]
_HARD_QD = re.compile(
    r'QUYẾT ĐỊNH\s*(ĐÌNH CHỈ|V/V|SỐ\s*\d|XÉT XỬ|CÔNG NHẬN)',
    re.IGNORECASE
)
_DINH_CHI = ["ĐÌNH CHỈ VỤ ÁN", "ĐÌNH CHỈ XÉT XỬ", "ĐÌNH CHỈ PHIÊN TÒA"]

def _by_so_vb(so_vb: str) -> str | None:
    su = so_vb.upper()
    for code, lbl in _SPECIAL_CODES.items():
        if code in su: return lbl
    for pat, lbl in _KY_HIEU_MAP:
        if pat.search(so_vb): return lbl
    return None

def _is_hard_qd(head: str) -> bool:
    h = head.upper()  # head đã ngắn, upper OK
    return _HARD_QD.search(h) is not None or any(p in h for p in _DINH_CHI)

def _by_keywords(text: str) -> tuple[str | None, float]:
    # FIX #9: cắt trước rồi upper
    t2000 = text[:2000].upper()
    t500  = t2000[:500]

    scores = dict.fromkeys(LABEL_SET, 0)
    is_qd_title = (bool(re.search(r'QUYẾT ĐỊNH\s*[\n\r]', t500))
                   or _is_hard_qd(t500))

    if is_qd_title:
        scores["Quyết định"] += 5
        for kw, lbl, w in _KEYWORDS:
            if lbl in ("Bản án hình sự", "Bản án dân sự"): continue
            if kw in t2000: scores[lbl] += w
    else:
        if re.search(r'QUYẾT ĐỊNH\s{0,5}(V/V|SỐ|ĐÌNH CHỈ)', t500):
            scores["Quyết định"] += 5
        for kw, lbl, w in _KEYWORDS:
            if kw in t2000: scores[lbl] += w

    best = max(scores, key=scores.get)
    if scores[best] == 0: return None, 0.0
    vals   = sorted(scores.values(), reverse=True)
    margin = vals[0] - vals[1] if len(vals) > 1 else vals[0]
    return best, min(0.95, 0.6 + margin * 0.05)

def classify_rule(pages: list[dict]) -> tuple[str, float]:
    texts    = [p["text_full"] for p in pages[:3]]
    head500  = texts[0][:500] if texts else ""

    # Priority 0: guard cứng
    if _is_hard_qd(head500):
        return "Quyết định", 0.97

    # Priority 1: số ký hiệu (FIX #4: RE đã IGNORECASE)
    for t in texts:
        for sv in RE_SO_VAN_BAN.findall(t):
            lbl = _by_so_vb(sv)
            if lbl: return lbl, 0.98

    # Priority 2: keyword scoring
    lbl, conf = _by_keywords(" ".join(texts))
    if lbl: return lbl, conf

    # Fallback: trang 2
    if len(pages) > 1:
        t2 = pages[1]["text_full"]
        for sv in RE_SO_VAN_BAN.findall(t2):
            lbl = _by_so_vb(sv)
            if lbl: return lbl, 0.90
        lbl2, conf2 = _by_keywords(t2)
        if lbl2: return lbl2, max(0.5, conf2 - 0.1)

    return "Quyết định", 0.50

print("✅ Classifier v4 xong.")

✅ Classifier v4 xong.


## 🤖 Cell 6 — Qwen2.5 Verifier

In [22]:
QWEN_URL     = "http://localhost:11434/api/generate"
QWEN_MODEL   = "qwen2.5:7b"
QWEN_TIMEOUT = 30

_QWEN_PROMPT = (
    "Bạn là chuyên gia phân loại tài liệu pháp lý Việt Nam.\n"
    "Chỉ trả lời đúng 1 trong 4 nhãn (không giải thích, không dấu ngoặc):\n"
    "  Quyết định\n  Bản án dân sự\n  Bản án hình sự\n  Bản án hành chính"
)

def _qwen_classify(text_head: str) -> tuple[str | None, float]:
    prompt = f"{_QWEN_PROMPT}\n\n---\n{text_head[:800]}\n---\nNhãn:"
    try:
        resp = requests.post(QWEN_URL,
            json={"model": QWEN_MODEL, "prompt": prompt,
                  "stream": False, "options": {"temperature": 0}},
            timeout=QWEN_TIMEOUT)
        if resp.status_code != 200: return None, 0.0
        raw = resp.json().get("response", "").strip()
        for lbl in LABEL_SET:
            if lbl.lower() in raw.lower(): return lbl, 0.88
        return None, 0.0
    except Exception: return None, 0.0

def _qwen_available() -> bool:
    try: return requests.get("http://localhost:11434", timeout=3).status_code == 200
    except Exception: return False

QWEN_ON = _qwen_available()
print(f"🤖 Qwen2.5 = {'✅ online' if QWEN_ON else '⚠️  offline – chỉ rule-based'}")

🤖 Qwen2.5 = ⚠️  offline – chỉ rule-based


## 🚨 Cell 7 — Mislabel Detector (v4: flag Qwen≠rule kể cả conf cao)

In [23]:


def detect_mislabels(segments: list[dict], gt: dict | None = None) -> list[dict]:
    flagged = []
    for seg in segments:
        ps      = seg["page_start"]
        rule    = seg.get("type")
        qwen    = seg.get("qwen_type")
        conf    = seg.get("confidence", 1.0)
        gt_lbl  = gt.get(ps) if gt else None
        reasons = []

        # Rule vs Qwen bất đồng (mọi mức conf)
        if qwen and qwen in LABEL_SET and rule != qwen:
            severity = "HIGH" if conf >= 0.95 else "MED"
            reasons.append(f"[{severity}] rule={rule} ≠ qwen={qwen}")

        # Confidence thấp
        if conf < 0.70:
            reasons.append(f"low_conf={conf:.2f}")

        # So với GT
        if gt_lbl and gt_lbl != rule:
            reasons.append(f"gt={gt_lbl} ≠ pred={rule}")

        if reasons:
            flagged.append({
                "segment"   : seg.get("segment"),
                "pages"     : f"p{seg['page_start']}-{seg['page_end']}",
                "rule_label": rule, "qwen_label": qwen, "gt_label": gt_lbl,
                "confidence": conf, "reasons": reasons,
                "suggested" : gt_lbl or (qwen if qwen in LABEL_SET else rule),
            })
    return flagged

def print_mislabel_report(flagged: list[dict]):
    if not flagged:
        print("✅ Không phát hiện nhãn sai!")
        return
    high = [f for f in flagged if any("HIGH" in r for r in f["reasons"])]
    print(f"\n🚨 {len(flagged)} flag | {len(high)} HIGH severity")
    print("-" * 76)
    for f in flagged:
        sev_icon = "🔴" if any("HIGH" in r for r in f["reasons"]) else "🟡"
        print(f"  {sev_icon} Seg {f['segment']:>3} | {f['pages']:>12} | "
              f"rule={f['rule_label']:<22} qwen={str(f['qwen_label']):<20}")
        for r in f["reasons"]:
            print(f"         · {r}")
        print(f"         ➜ gợi ý: {f['suggested']}")
    print("-" * 76)

print("✅ Mislabel Detector v4 xong.")

✅ Mislabel Detector v4 xong.


## 🚀 Cell 8 — Full Pipeline (v4)

In [24]:
def full_pipeline(
    pdf_path         : str,
    output_dir       : str,
    cache_path       : str | None = None,
    gt_boundaries    : set  | None = None,
    gt_segment_types : dict | None = None,
    use_qwen         : bool = True,
    force_extract    : bool = False,
    use_timestamp_dir: bool = True,   # FIX #7
) -> dict:
    t0 = time.time()

    # FIX #7: subfolder timestamp tránh overwrite
    if use_timestamp_dir:
        run_dir = Path(output_dir) / datetime.now().strftime("%Y%m%d_%H%M%S")
    else:
        run_dir = Path(output_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    # ── 1. Extract ────────────────────────────────────────────
    print("📑 [1/5] Extracting text...")
    feats = batch_extract(pdf_path, cache_path=cache_path, force=force_extract)
    total_pages = len(feats)  # FIX #8: từ feats, không phải global

    # ── 2. Boundary ───────────────────────────────────────────
    print("🔍 [2/5] Detecting boundaries...")
    boundaries, separators = compute_boundaries(feats)
    raw_segs = group_segments(feats, boundaries, separators)
    segs     = merge_short_segments(raw_segs)   # FIX #3
    print(f"   → {len(raw_segs)} raw → {len(segs)} sau merge")

    # ── 3. Classify ───────────────────────────────────────────
    print("🏷️  [3/5] Classifying...")
    results = []
    for i, pages in enumerate(segs):
        rule_lbl, rule_conf = classify_rule(pages)

        qwen_lbl = None
        if use_qwen and QWEN_ON:
            head     = " ".join(p["text_full"] for p in pages[:2])[:1200]
            qwen_lbl, _ = _qwen_classify(head)

        # Chọn nhãn cuối: Qwen thắng chỉ khi rule không tự tin
        if qwen_lbl and qwen_lbl in LABEL_SET and rule_conf < 0.80:
            final_lbl, final_conf = qwen_lbl, 0.88
        else:
            final_lbl, final_conf = rule_lbl, rule_conf

        results.append({
            "segment"   : i + 1,
            "page_start": pages[0]["page_num"],
            "page_end"  : pages[-1]["page_num"],
            "n_pages"   : len(pages),
            "type"      : final_lbl,
            "rule_type" : rule_lbl,
            "qwen_type" : qwen_lbl,
            "confidence": round(final_conf, 3),
        })

    # ── 4. Mislabel ───────────────────────────────────────────
    print("🚨 [4/5] Checking mislabels...")
    flagged = detect_mislabels(results, gt=gt_segment_types)
    print_mislabel_report(flagged)

    # ── 5. Split & Save ───────────────────────────────────────
    print("✂️  [5/5] Splitting PDFs...")
    src      = fitz.open(pdf_path)
    doc_id   = str(uuid.uuid4())[:8]
    sub_docs = []

    for r in results:
        lbl_safe = r["type"].replace(" ", "_")
        fname    = f"sub_{r['segment']:03d}_{lbl_safe}_p{r['page_start']}-{r['page_end']}.pdf"
        out_pdf  = fitz.open()
        out_pdf.insert_pdf(src, from_page=r["page_start"]-1, to_page=r["page_end"]-1)
        out_pdf.save(str(run_dir / fname))
        out_pdf.close()
        sub_docs.append({"type": r["type"],
                         "page_start": r["page_start"],
                         "page_end"  : r["page_end"]})
    src.close()

    elapsed  = int((time.time() - t0) * 1000)
    avg_conf = round(sum(r["confidence"] for r in results) / max(len(results), 1), 3)

    payload = {
        "document_id"       : doc_id,
        "classification"    : "Tài liệu hỗn hợp",
        # FIX #8: total_pages từ feats
        "summary"           : f"{len(sub_docs)} tài liệu | {total_pages} trang | {elapsed}ms",
        "confidence_overall": avg_conf,
        "processing_ms"     : elapsed,
        "mislabel_flags"    : len(flagged),
        "output_dir"        : str(run_dir),
        "metadata"          : [],
        "sub_documents"     : sub_docs,
    }

    json_path = run_dir / f"result_{doc_id}.json"
    with open(json_path, "w", encoding="utf-8") as fj:
        json.dump(payload, fj, ensure_ascii=False, indent=2)

    print(f"\n✅ Hoàn tất | {len(sub_docs)} segs | conf={avg_conf} | {elapsed}ms")
    print(f"   Output: {run_dir}")
    return payload

# import cần thêm cho FIX #7
from datetime import datetime
print("✅ Full Pipeline v4 xong.")

✅ Full Pipeline v4 xong.


## ▶️ Cell 9 — Chạy

In [25]:
result = full_pipeline(
    pdf_path         = PDF_PATH,
    output_dir       = OUTPUT_DIR,
    cache_path       = CACHE_PATH,
    gt_boundaries    = GT_BOUNDARIES,
    gt_segment_types = GT_SEGMENT_TYPES,
    use_qwen         = True,
    force_extract    = False,      # True = bỏ cache, extract lại
    use_timestamp_dir= True,       # False = overwrite (tiện debug)
)
print(json.dumps(result, ensure_ascii=False, indent=2))

📑 [1/5] Extracting text...
  [Pass 1] Text layer (ThreadPool)...
  → 109 trang | 1.8s
  → 77 text-layer | 32 scan (OCR)
  [Pass 2] Render 32 scan pages (main process)...


  Render pixmap:   0%|          | 0/32 [00:00<?, ?pg/s]

  [Pass 2] OCR (ProcessPool, 4 workers)...


  OCR:   0%|          | 0/32 [00:00<?, ?pg/s]

  → OCR xong: 158.5s
  ✅ Extract: 174.4s
  💾 Cache lưu: /content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/cache_features.pkl
🔍 [2/5] Detecting boundaries...
   → 28 raw → 28 sau merge
🏷️  [3/5] Classifying...
🚨 [4/5] Checking mislabels...

🚨 1 flag | 0 HIGH severity
----------------------------------------------------------------------------
  🟡 Seg   7 |       p29-34 | rule=Bản án hình sự         qwen=None                
         · gt=Quyết định ≠ pred=Bản án hình sự
         ➜ gợi ý: Quyết định
----------------------------------------------------------------------------
✂️  [5/5] Splitting PDFs...

✅ Hoàn tất | 28 segs | conf=0.968 | 174976ms
   Output: /content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output/20260604_012151
{
  "document_id": "4aed4672",
  "classification": "Tài liệu hỗn hợp",
  "summary": "28 tài liệu | 109 trang | 174976ms",
  "confidence_overall": 0.968,
  "processing_ms": 174976,
  "mislabel_flags": 1,
  "outp

## 📊 Cell 10 — Đánh giá Accuracy

In [26]:
def evaluate(sub_docs: list[dict], gt_types: dict) -> float:
    matched = total = 0
    print(f"\n{'Seg':>4} {'Pages':>12} {'Predict':>22} {'GT':>22} {'OK?':>5}")
    print("-" * 70)
    for i, s in enumerate(sub_docs, 1):
        gt = gt_types.get(s["page_start"])
        if gt is None: continue
        ok = s["type"] == gt
        matched += ok; total += 1
        print(f"{i:>4} p{s['page_start']}-{s['page_end']:>3}  "
              f"{s['type']:>22}  {gt:>22}  {'✅' if ok else '❌'}")
    acc = matched / total * 100 if total else 0
    print(f"\nAccuracy: {matched}/{total} = {acc:.1f}%")
    return acc

if 'result' in dir():
    evaluate(result["sub_documents"], GT_SEGMENT_TYPES)


 Seg        Pages                Predict                     GT   OK?
----------------------------------------------------------------------
   1 p1-  1              Quyết định              Quyết định  ✅
   4 p4-  6              Quyết định              Quyết định  ✅
   5 p7-  8              Quyết định              Quyết định  ✅
   6 p9- 28           Bản án dân sự           Bản án dân sự  ✅
   7 p29- 34          Bản án hình sự              Quyết định  ❌
   8 p35- 42           Bản án dân sự           Bản án dân sự  ✅
   9 p43- 43          Bản án hình sự          Bản án hình sự  ✅
  10 p44- 49          Bản án hình sự          Bản án hình sự  ✅
  11 p50- 51              Quyết định              Quyết định  ✅
  12 p52- 52              Quyết định              Quyết định  ✅
  14 p54- 54              Quyết định              Quyết định  ✅
  17 p57- 58              Quyết định              Quyết định  ✅
  18 p59- 59              Quyết định              Quyết định  ✅
  20 p62- 62              Quyế